# Generalization-aligned feature-selection analysis

This notebook explains the architectural failure in 2.1 and inspects the fold-level evidence produced by 2.2. It is intentionally read-only: the canonical selection is executed by `run_selection.py`.


## Locate the reproducible experiment artifacts

Resolve the repository root and load only versioned CSV/JSON artifacts. Missing full-run artifacts are reported without silently substituting smoke results.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "data" / "splits").is_dir():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT))
EXP_DIR = PROJECT_ROOT / "notebooks/experiment/derived_8.2-feature-selection-2.2"
ARTIFACT_ROOT = EXP_DIR / "artifacts/final"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Full artifacts available:", ARTIFACT_ROOT.exists())

PROJECT_ROOT: /scratch/user/u.rp352032/MDR-Project
Full artifacts available: True


## Audit fold construction and selected-set uncertainty

The table exposes the validation year, held-out stations, normalized error, and candidate size. This is the evidence that replaces pooled XGBoost gain and row-bootstrap frequency.

In [2]:
fold_paths = sorted(ARTIFACT_ROOT.glob("*/global/fold_metrics.csv"))
if fold_paths:
    fold_metrics = pd.concat(
        [pd.read_csv(path).assign(dataset=path.parents[1].name) for path in fold_paths],
        ignore_index=True,
    )
    display(
        fold_metrics.groupby(["dataset", "n_features"])
        .agg(mean_nrmse=("nrmse", "mean"), std_nrmse=("nrmse", "std"), folds=("fold_id", "nunique"))
        .reset_index()
    )
else:
    print("Run run_selection.py to generate the full fold audit.")

,dataset,n_features,mean_nrmse,std_nrmse,folds
0,derived_8.0,40,0.574915,0.171662,32
1,derived_8.0,50,0.574011,0.162986,32
2,derived_8.0,65,0.581374,0.164663,32
3,derived_8.0,80,0.575180,0.170703,32
4,derived_8.0,100,0.579998,0.189188,32
5,derived_8.2,40,0.762142,0.165447,32
6,derived_8.2,50,0.752254,0.165364,32
7,derived_8.2,65,0.773630,0.174880,32
8,derived_8.2,80,0.784025,0.178284,32
9,derived_8.2,100,0.683491,0.151420,32
